In [1]:
import json
import pandas as pd
from datetime import datetime
import os

# 원하는 시나리오 가져오기

# file_path = "../../edge/processed_topic_samples/plc_processed_cnc_toolchange_s01.json"
file_path = "../../edge/processed_topic_samples/plc_processed_cnc_tooldelay_s02.json"
# file_path = "../../edge/processed_topic_samples/plc_processed_line_jam_s03.json"


with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

for obj in data:
    ts_str = obj.get("timestamp")
    if ts_str:
        obj["timestamp"] = datetime.fromisoformat(ts_str.replace("Z", "+00:00"))
    else:
        obj["timestamp"] = None

df = pd.DataFrame(data)

df = df[df["timestamp"].notna()].copy()
df.sort_values("timestamp", inplace=True)
df.reset_index(drop=True, inplace=True)

df.head()

,timestamp,edgeId,equipmentId,jobId,productId,lotId,eventType,previousState,currentState,previousMode,currentMode,durationSec,alarmCode,deltaTotal,deltaGood,deltaReject
0,2026-03-03 08:00:09+00:00,EDGE_01,CNC_01,JOB_001,BRAKEROTER_A,LOT_01,MODE_TRANSITION,NaN,NaN,SETUP,AUTO,NaN,NaN,NaN,NaN,NaN
1,2026-03-03 08:00:10+00:00,EDGE_01,CNC_01,JOB_001,BRAKEROTER_A,LOT_01,STATE_TRANSITION,IDLE,RUN,NaN,NaN,10.0,NaN,NaN,NaN,NaN
2,2026-03-03 08:00:11+00:00,EDGE_01,Conveyor_01,JOB_001,BRAKEROTER_A,LOT_01,MODE_TRANSITION,NaN,NaN,SETUP,AUTO,NaN,NaN,NaN,NaN,NaN
3,2026-03-03 08:00:12+00:00,EDGE_01,Conveyor_01,JOB_001,BRAKEROTER_A,LOT_01,STATE_TRANSITION,IDLE,RUN,NaN,NaN,10.0,NaN,NaN,NaN,NaN
4,2026-03-03 08:02:40+00:00,EDGE_01,CNC_01,JOB_001,BRAKEROTER_A,LOT_01,STATE_TRANSITION,RUN,COMPLETE,NaN,NaN,150.0,NaN,NaN,NaN,NaN


In [2]:
# 라인 전체 Alarm Timeline 생성
df_alarms = df[df["eventType"].isin(["ALARM_ON", "ALARM_OFF"])][
    ["timestamp", "equipmentId", "alarmCode", "eventType"]
].copy()

df_alarms.sort_values("timestamp", inplace=True)

active_alarms = []
alarm_timeline = []

for _, row in df_alarms.iterrows():

    if row["eventType"] == "ALARM_ON":
        active_alarms.append({
            "equipmentId": row["equipmentId"],
            "alarmCode": row["alarmCode"]
        })

    elif row["eventType"] == "ALARM_OFF":
        active_alarms = [
            a for a in active_alarms
            if not (
                a["equipmentId"] == row["equipmentId"]
                and a["alarmCode"] == row["alarmCode"]
            )
        ]

    alarm_timeline.append({
        "timestamp": row["timestamp"],
        "activeAlarms": active_alarms.copy()
    })

df_alarm_timeline = pd.DataFrame(alarm_timeline)

df_alarm_timeline.head()

,timestamp,activeAlarms
0,2026-03-03 08:07:00+00:00,"[{'equipmentId': 'CNC_01', 'alarmCode': 11.0}]"
1,2026-03-03 08:32:30+00:00,[]


In [3]:
# 특정 시점의 활성 Alarm 조회 함수
def get_active_alarms(ts):

    rows = df_alarm_timeline[
        df_alarm_timeline["timestamp"] <= ts
    ]

    if rows.empty:
        return []

    return rows.iloc[-1]["activeAlarms"]

In [4]:
# cnc 이벤트만 추출
df_cnc = df[df["equipmentId"] == "CNC_01"].copy()

df_cnc.sort_values("timestamp", inplace=True)
df_cnc.reset_index(drop=True, inplace=True)

df_cnc.head()

,timestamp,edgeId,equipmentId,jobId,productId,lotId,eventType,previousState,currentState,previousMode,currentMode,durationSec,alarmCode,deltaTotal,deltaGood,deltaReject
0,2026-03-03 08:00:09+00:00,EDGE_01,CNC_01,JOB_001,BRAKEROTER_A,LOT_01,MODE_TRANSITION,NaN,NaN,SETUP,AUTO,NaN,NaN,NaN,NaN,NaN
1,2026-03-03 08:00:10+00:00,EDGE_01,CNC_01,JOB_001,BRAKEROTER_A,LOT_01,STATE_TRANSITION,IDLE,RUN,NaN,NaN,10.0,NaN,NaN,NaN,NaN
2,2026-03-03 08:02:40+00:00,EDGE_01,CNC_01,JOB_001,BRAKEROTER_A,LOT_01,STATE_TRANSITION,RUN,COMPLETE,NaN,NaN,150.0,NaN,NaN,NaN,NaN
3,2026-03-03 08:02:50+00:00,EDGE_01,CNC_01,JOB_001,BRAKEROTER_A,LOT_01,STATE_TRANSITION,COMPLETE,RUN,NaN,NaN,10.0,NaN,NaN,NaN,NaN
4,2026-03-03 08:05:20+00:00,EDGE_01,CNC_01,JOB_001,BRAKEROTER_A,LOT_01,STATE_TRANSITION,RUN,COMPLETE,NaN,NaN,150.0,NaN,NaN,NaN,NaN


In [5]:
# State / Mode 채우기 (processed에는 나눠져 있어서 State Reconstruction 필요)
current_state = None
current_mode = None

filled_states = []
filled_modes = []

for _, row in df_cnc.iterrows():

    if row["eventType"] == "STATE_TRANSITION":
        current_state = row["currentState"]

    if row["eventType"] == "MODE_TRANSITION":
        current_mode = row["currentMode"]

    filled_states.append(current_state)
    filled_modes.append(current_mode)

df_cnc["state_filled"] = filled_states
df_cnc["mode_filled"] = filled_modes

df_cnc.head()

,timestamp,edgeId,equipmentId,jobId,productId,lotId,eventType,previousState,currentState,previousMode,currentMode,durationSec,alarmCode,deltaTotal,deltaGood,deltaReject,state_filled,mode_filled
0,2026-03-03 08:00:09+00:00,EDGE_01,CNC_01,JOB_001,BRAKEROTER_A,LOT_01,MODE_TRANSITION,NaN,NaN,SETUP,AUTO,NaN,NaN,NaN,NaN,NaN,NaN,AUTO
1,2026-03-03 08:00:10+00:00,EDGE_01,CNC_01,JOB_001,BRAKEROTER_A,LOT_01,STATE_TRANSITION,IDLE,RUN,NaN,NaN,10.0,NaN,NaN,NaN,NaN,RUN,AUTO
2,2026-03-03 08:02:40+00:00,EDGE_01,CNC_01,JOB_001,BRAKEROTER_A,LOT_01,STATE_TRANSITION,RUN,COMPLETE,NaN,NaN,150.0,NaN,NaN,NaN,NaN,COMPLETE,AUTO
3,2026-03-03 08:02:50+00:00,EDGE_01,CNC_01,JOB_001,BRAKEROTER_A,LOT_01,STATE_TRANSITION,COMPLETE,RUN,NaN,NaN,10.0,NaN,NaN,NaN,NaN,RUN,AUTO
4,2026-03-03 08:05:20+00:00,EDGE_01,CNC_01,JOB_001,BRAKEROTER_A,LOT_01,STATE_TRANSITION,RUN,COMPLETE,NaN,NaN,150.0,NaN,NaN,NaN,NaN,COMPLETE,AUTO


In [6]:
# 다음 이벤트까지 Duration 계산
df_cnc["next_timestamp"] = df_cnc["timestamp"].shift(-1)

df_cnc["durationSec"] = (
    df_cnc["next_timestamp"] - df_cnc["timestamp"]
).dt.total_seconds()

df_cnc.head()

,timestamp,edgeId,equipmentId,jobId,productId,lotId,eventType,previousState,currentState,previousMode,currentMode,durationSec,alarmCode,deltaTotal,deltaGood,deltaReject,state_filled,mode_filled,next_timestamp
0,2026-03-03 08:00:09+00:00,EDGE_01,CNC_01,JOB_001,BRAKEROTER_A,LOT_01,MODE_TRANSITION,NaN,NaN,SETUP,AUTO,1.0,NaN,NaN,NaN,NaN,NaN,AUTO,2026-03-03 08:00:10+00:00
1,2026-03-03 08:00:10+00:00,EDGE_01,CNC_01,JOB_001,BRAKEROTER_A,LOT_01,STATE_TRANSITION,IDLE,RUN,NaN,NaN,150.0,NaN,NaN,NaN,NaN,RUN,AUTO,2026-03-03 08:02:40+00:00
2,2026-03-03 08:02:40+00:00,EDGE_01,CNC_01,JOB_001,BRAKEROTER_A,LOT_01,STATE_TRANSITION,RUN,COMPLETE,NaN,NaN,10.0,NaN,NaN,NaN,NaN,COMPLETE,AUTO,2026-03-03 08:02:50+00:00
3,2026-03-03 08:02:50+00:00,EDGE_01,CNC_01,JOB_001,BRAKEROTER_A,LOT_01,STATE_TRANSITION,COMPLETE,RUN,NaN,NaN,150.0,NaN,NaN,NaN,NaN,RUN,AUTO,2026-03-03 08:05:20+00:00
4,2026-03-03 08:05:20+00:00,EDGE_01,CNC_01,JOB_001,BRAKEROTER_A,LOT_01,STATE_TRANSITION,RUN,COMPLETE,NaN,NaN,10.0,NaN,NaN,NaN,NaN,COMPLETE,AUTO,2026-03-03 08:05:30+00:00


In [7]:
# Timeline Dataset 생성 (시각화용)
timeline_df = df_cnc.dropna(subset=["durationSec"]).copy()

timeline_df = timeline_df[[
    "timestamp",
    "next_timestamp",
    "state_filled",
    "mode_filled",
    "durationSec"
]]

timeline_df.rename(columns={
    "timestamp": "start",
    "next_timestamp": "end",
    "state_filled": "state",
    "mode_filled": "mode"
}, inplace=True)

timeline_df.head()


,start,end,state,mode,durationSec
0,2026-03-03 08:00:09+00:00,2026-03-03 08:00:10+00:00,NaN,AUTO,1.0
1,2026-03-03 08:00:10+00:00,2026-03-03 08:02:40+00:00,RUN,AUTO,150.0
2,2026-03-03 08:02:40+00:00,2026-03-03 08:02:50+00:00,COMPLETE,AUTO,10.0
3,2026-03-03 08:02:50+00:00,2026-03-03 08:05:20+00:00,RUN,AUTO,150.0
4,2026-03-03 08:05:20+00:00,2026-03-03 08:05:30+00:00,COMPLETE,AUTO,10.0


In [8]:
# 활성 알람컬럼 추가
timeline_df["alarmCodes"] = timeline_df["start"].apply(get_active_alarms)
df_prod = df[df["eventType"] == "PRODUCTION_COMPLETE"]

tracks =[]

# state track
for _, row in timeline_df.iterrows():
    tracks.append({
        "track":"STATE",
        "start": row["start"],
        "end": row["end"],
        "label": row["state"],
        "mode": row["mode"],
        "alarm": row["alarmCodes"]
    })

# mode track
for _, row in timeline_df.iterrows():

    if pd.notna(row["mode"]):

        tracks.append({
            "track": "MODE",
            "start": row["start"],
            "end": row["end"],
            "label": row["mode"],
            "mode": row["mode"],
            "alarm": row["alarmCodes"]
        })

# alarm Track
for _, row in timeline_df.iterrows():

    alarms = row["alarmCodes"]
    if isinstance(alarms, list) and len(alarms) > 0:

        tracks.append({
            "track": "ALARM",
            "start": row["start"],
            "end": row["end"],
            "label": ",".join(map(str,row["alarmCodes"])),
            "mode": row["mode"],
            "alarm": row["alarmCodes"]
        })

# Good/Result 결과 Track
for _, row in df_prod.iterrows():
    if row.get("eventType") == "PRODUCTION_COMPLETE":
        if row.get("deltaGood") == 1:  # 1이면 GOOD
            label = "GOOD"
        elif row.get("deltaReject") == 1:  # 1이면 REJECT
            label = "REJECT"
        else:
            continue  # deltaGood/deltaReject 둘 다 1이 아니면 skip
        tracks.append({
            "track": "RESULT",
            "start": row["timestamp"],
            "end": row["timestamp"],  # point 이벤트이므로 start=end
            "label": label,
            "mode": None,
            "alarm": [],
            "marker_size" : 12
        })

df_tracks = pd.DataFrame(tracks)
df_tracks.head()


,track,start,end,label,mode,alarm,marker_size
0,STATE,2026-03-03 08:00:09+00:00,2026-03-03 08:00:10+00:00,NaN,AUTO,[],NaN
1,STATE,2026-03-03 08:00:10+00:00,2026-03-03 08:02:40+00:00,RUN,AUTO,[],NaN
2,STATE,2026-03-03 08:02:40+00:00,2026-03-03 08:02:50+00:00,COMPLETE,AUTO,[],NaN
3,STATE,2026-03-03 08:02:50+00:00,2026-03-03 08:05:20+00:00,RUN,AUTO,[],NaN
4,STATE,2026-03-03 08:05:20+00:00,2026-03-03 08:05:30+00:00,COMPLETE,AUTO,[],NaN


In [9]:
import plotly.express as px
import plotly.graph_objects as go

# timeline 막대 생성 (STATE / MODE / ALARM)
fig = px.timeline(
    df_tracks[df_tracks["track"].isin(["STATE","MODE","ALARM"])],
    x_start="start",
    x_end="end",
    y="track",
    color="label",
    hover_data=["label","mode","alarm"],
    title="CNC MES Timeline"
)

fig.update_yaxes(autorange="reversed")
fig.update_layout(
    height=500,
    width=1500,
    margin=dict(l=50, r=50, t=80, b=50)
)
fig.update_xaxes(
    rangeslider_visible=True,
    showgrid=True,
    tickformat="%H:%M:%S",
    hoverformat="%Y-%m-%d %H:%M:%S"
)

# RESULT 포인트 추가 (GOOD/REJECT 구분만)
df_result = df_tracks[df_tracks["track"] == "RESULT"]
if not df_result.empty:
    fig.add_trace(
        go.Scatter(
            x=df_result["start"].tolist(),
            y=df_result["track"].tolist(),
            mode="markers+text",
            marker=dict(
                size=df_result["marker_size"].fillna(12).tolist(),
                color=df_result["label"].map({"GOOD":"green","REJECT":"red"}).tolist()
            ),
            text=df_result["label"].tolist(),  # GOOD/REJECT 표시
            textposition="top center",
            name="RESULT",
            hovertemplate="Time: %{x}<br>Status: %{text}<extra></extra>"
        )
    )

fig.show()

In [10]:
import plotly.io as pio

# 인터랙티브 HTML로 저장
pio.write_html(fig, file ="tooldelay_s02_timeline.html", include_plotlyjs='cdn', full_html=True)